# 16 — Probability and Statistics

**Master syllabus coverage:** V2 2.7 plus covariance

## Why this matters

Language models represent conditional probability distributions, while experiments estimate uncertain quantities from finite samples. Probability describes the model; statistics keeps conclusions honest.

## Learning objectives

- Validate, sample, and summarize a discrete distribution.
- Compute conditional probability from a joint table.
- Interpret variance, covariance, and correlation.
- Measure how sample-size changes estimator variability.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Random variables and distributions

A discrete distribution assigns nonnegative mass $p(x)$ that sums to one. A random
variable is a numerical function of an outcome. Its expectation is a probability-
weighted average, not necessarily a value the variable can actually take:

$$\mathbb{E}[X]=\sum_x x\,p(x),\qquad
\operatorname{Var}(X)=\mathbb{E}[(X-\mathbb{E}[X])^2].$$


In [ ]:
import numpy as np
import torch

rng = np.random.default_rng(42)
outcomes = np.array([0.0, 1.0, 2.0])
probabilities = np.array([0.2, 0.5, 0.3])
assert np.all(probabilities >= 0) and np.isclose(probabilities.sum(), 1.0)

expectation = np.sum(outcomes * probabilities)
variance = np.sum((outcomes - expectation) ** 2 * probabilities)
samples = rng.choice(outcomes, size=100_000, p=probabilities)
print("exact mean/variance:", expectation, variance)
print("sample mean/variance:", samples.mean(), samples.var())


## 2. Conditional probability and Bayes' rule

$$p(A\mid B)=\frac{p(A,B)}{p(B)},\qquad
p(A\mid B)=\frac{p(B\mid A)p(A)}{p(B)}.$$

Conditioning changes the relevant population. Language models are conditional
distributions: the same next token can receive very different probability under a
different prefix.


In [ ]:
# Rows: context is question / statement. Columns: next token is '?' / '.'.
joint = np.array([[0.30, 0.10], [0.05, 0.55]])
assert np.isclose(joint.sum(), 1.0)
context_mass = joint.sum(axis=1, keepdims=True)
conditional = joint / context_mass
print("p(punctuation | context):\n", conditional)
np.testing.assert_allclose(conditional.sum(axis=1), 1.0)


## 3. Covariance and correlation

$$\operatorname{Cov}(X,Y)=\mathbb{E}[(X-\mu_X)(Y-\mu_Y)]$$

Positive covariance means variables tend to move together; negative means opposite
movement. Correlation divides by standard deviations. Neither implies causation, and
zero covariance does not generally imply independence.


In [ ]:
x = rng.normal(size=10_000)
y = 2.0 * x + rng.normal(scale=0.5, size=10_000)
z = x**2
covariance = np.cov(np.stack([x, y]), ddof=0)
correlation = np.corrcoef(x, y)[0, 1]
print("covariance matrix:\n", covariance)
print("corr(x,y):", correlation, "corr(x,x²):", np.corrcoef(x, z)[0, 1])
assert correlation > 0.9


## 4. Sampling error and the law of large numbers

A finite sample statistic varies across datasets. As independent sample size grows,
the sample mean concentrates around the population expectation, with standard error
proportional to $1/\sqrt{n}$. One seed is one draw, not proof of a general behavior.


In [ ]:
true_mean = expectation
for n in (10, 100, 1_000, 10_000):
    estimates = np.array([
        rng.choice(outcomes, size=n, p=probabilities).mean() for _ in range(200)
    ])
    print(f"n={n:5}: estimate mean={estimates.mean():.3f}, std across trials={estimates.std():.4f}")

torch.manual_seed(42)
categorical = torch.distributions.Categorical(probs=torch.tensor(probabilities))
torch_samples = categorical.sample((10_000,))
assert torch_samples.shape == (10_000,)


## Failure modes to recognize

- **Invalid probability vector:** negative mass or a sum different from one breaks sampling semantics.
- **Conditioning on the wrong axis:** rows normalize while the intended event lives in columns.
- **Correlation as causation:** a predictive relationship is mistaken for a mechanism.
- **Single-run certainty:** noise from dataset sampling or initialization is ignored.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Compute a posterior with Bayes' rule for a diagnostic test with an imbalanced prior.
2. Construct dependent random variables with approximately zero correlation.
3. Empirically verify that standard error shrinks near `1/sqrt(n)`.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can move between joint, marginal, and conditional distributions and quantify uncertainty in a sample estimate.

**Next:** 17 — Likelihood and information.
